<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ VoxCPM2 — Multilanguage TTS</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab Edition — Created by <strong>TheBlackBoxSecurity</strong></h3>
  <p style='color: #ddd; margin: 0 0 0 0;'>Google Colab GPU | Python 3.12 · VoxCPM2 2B · 30 Languages · 48kHz Output · Voice Design & Cloning</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/TheBlackBoxSecurity-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@TheBlackBoxSecurity?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/TheBlackBoxSecurity">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### ⚠️ Quick Start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. In **Runtime version**, select a Python **3.12** runtime (for example **2026.07** or another available 3.12 runtime)
3. Run all cells in order
4. The setup cell pins compatible dependencies before importing VoxCPM2
5. The Gradio UI link will appear at the bottom

In [ ]:
#@title 📦 Install Dependencies — Stable Colab Environment

import sys
import subprocess

# VoxCPM2 officially requires Python >= 3.10 and < 3.13.
PY_MAJOR = sys.version_info.major
PY_MINOR = sys.version_info.minor

print(f"🐍 Python: {sys.version.split()[0]}")

if (PY_MAJOR, PY_MINOR) < (3, 10) or (PY_MAJOR, PY_MINOR) >= (3, 13):
    raise RuntimeError(
        "\n❌ Unsupported Python runtime.\n"
        f"Detected: Python {PY_MAJOR}.{PY_MINOR}\n\n"
        "VoxCPM2 requires Python >= 3.10 and < 3.13.\n"
        "In Google Colab, use:\n"
        "Runtime → Change runtime type → Runtime version → a Python 3.12 runtime "
        "(for example 2026.07, if available).\n"
        "Then run this notebook again."
    )

# IMPORTANT:
# Colab runtimes ship with NumPy 2.x.  This notebook intentionally pins
# NumPy to 1.26.4 to avoid the NumPy ABI mismatch seen with this TTS stack.
#
# VoxCPM is also pinned to the current 2.0.3 release so that a future PyPI
# update cannot silently change the model package underneath this notebook.

PACKAGES = [
    "numpy==1.26.4",
    "voxcpm==2.0.3",
    "gradio",
    "soundfile",
]

print("📦 Installing pinned dependencies...")
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        *PACKAGES,
    ]
)

# Re-assert NumPy after dependency resolution.
# This is intentionally done last so another dependency cannot silently
# upgrade NumPy back to 2.x during installation.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--force-reinstall",
        "numpy==1.26.4",
    ]
)

# Verify the installed distribution without importing any compiled TTS
# dependencies before the environment is ready.
import importlib.metadata as metadata

numpy_version = metadata.version("numpy")
voxcpm_version = metadata.version("voxcpm")

print(f"✅ NumPy:  {numpy_version}")
print(f"✅ VoxCPM: {voxcpm_version}")

if numpy_version != "1.26.4":
    raise RuntimeError(
        f"❌ NumPy pin failed. Expected 1.26.4, got {numpy_version}."
    )

print("\n" + "=" * 62)
print("✅ Dependencies installed successfully.")
print("➡️ Next: run the 'Verify GPU & Environment' cell.")
print("=" * 62)


In [ ]:
#@title ✅ Verify GPU & Environment

import sys
import importlib.metadata as metadata

print("=" * 62)
print("VoxCPM2 Environment Check")
print("=" * 62)

print(f"🐍 Python : {sys.version.split()[0]}")

numpy_version = metadata.version("numpy")
voxcpm_version = metadata.version("voxcpm")

print(f"🔢 NumPy  : {numpy_version}")
print(f"📦 VoxCPM : {voxcpm_version}")

# NumPy compatibility check
if numpy_version != "1.26.4":
    raise RuntimeError(
        f"❌ NumPy {numpy_version} detected. "
        "This notebook requires NumPy 1.26.4."
    )

import numpy as np
import torch

print(f"🔥 PyTorch: {torch.__version__}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No CUDA GPU detected!\n"
        "Go to Runtime → Change runtime type → Hardware accelerator → GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
cuda_version = torch.version.cuda

print(f"🎮 GPU    : {gpu_name}")
print(f"💾 VRAM   : {vram_gb:.1f} GB")
print(f"⚡ CUDA   : {cuda_version}")

print("=" * 62)
print("✅ Environment looks good!")
print("=" * 62)


In [ ]:
#@title 🚀 Load VoxCPM2 Model

import gc
import numpy as np
import torch
from voxcpm import VoxCPM

# Keep these settings explicit so the notebook is easy to maintain.
MODEL_ID = "openbmb/VoxCPM2"
LOAD_DENOISER = True
OPTIMIZE = False  # Safer for Colab/debugging; avoids torch.compile issues.

print("🚀 Loading VoxCPM2...")
print(f"Model : {MODEL_ID}")
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

try:
    model = VoxCPM.from_pretrained(
        MODEL_ID,
        load_denoiser=LOAD_DENOISER,
        optimize=OPTIMIZE,
        device="cuda",
    )
except TypeError:
    # Compatibility fallback for package builds that do not expose device=.
    model = VoxCPM.from_pretrained(
        MODEL_ID,
        load_denoiser=LOAD_DENOISER,
        optimize=OPTIMIZE,
    )

SAMPLE_RATE = model.tts_model.sample_rate

print(f"✅ Model loaded!")
print(f"🔊 Sample rate: {SAMPLE_RATE} Hz")

# ------------------------------------------------------------------
# Warm-up
# ------------------------------------------------------------------
# VoxCPM2's current Python API returns a waveform from generate().
# Older builds/workflows may return an iterator of chunks. The helper
# below supports both forms.
print("🔥 Warming up...")

def collect_audio(result) -> np.ndarray:
    """Convert VoxCPM output (array/tensor/iterator) into one 1-D NumPy array."""
    if torch.is_tensor(result):
        arr = result.detach().float().cpu().numpy()
        return np.asarray(arr).reshape(-1)

    if isinstance(result, np.ndarray):
        return np.asarray(result).reshape(-1)

    # Some older/alternate implementations return an iterable of chunks.
    chunks = []
    for chunk in result:
        if torch.is_tensor(chunk):
            chunk = chunk.detach().float().cpu().numpy()
        arr = np.asarray(chunk)
        if arr.ndim == 0:
            arr = arr.reshape(1)
        else:
            arr = arr.reshape(-1)
        chunks.append(arr)

    if not chunks:
        raise RuntimeError("Model returned no audio.")

    return np.concatenate(chunks)

with torch.inference_mode():
    warmup_result = model.generate(
        text="Hello, warm up test.",
        inference_timesteps=5,
        seed=42,
    )

warmup_wav = collect_audio(warmup_result)

used = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"✅ Warm-up complete!")
print(f"🎯 Warm-up samples: {len(warmup_wav):,}")
print(f"💾 GPU memory: {used:.1f} GB allocated / {reserved:.1f} GB reserved / {total:.1f} GB total")

del warmup_result, warmup_wav
gc.collect()
torch.cuda.empty_cache()

print("✅ VoxCPM2 is ready.")


In [ ]:
#@title 🎛️ Launch Gradio Interface

import gradio as gr
import soundfile as sf
import numpy as np
import tempfile
import os
import random
import torch

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎙️ VoxCPM2 — Multilingual TTS</div>
  <div class="brand-subtitle">Created by <strong>The BlackBox Security</strong> &nbsp;|&nbsp; 2B Parameters · 30 Languages · 48kHz Output · Voice Design &amp; Cloning</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@TheBlackBoxSecurity?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/TheBlackBoxSecurity" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://TheBlackBoxSecurity.com" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

# ── Helpers ───────────────────────────────────────────────────────

def save_wav(wav_array: np.ndarray) -> str:
    wav_array = np.asarray(wav_array).reshape(-1)
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    tmp.close()
    sf.write(tmp.name, wav_array, SAMPLE_RATE)
    return tmp.name


def resolve_seed(seed, locked):
    if locked and seed is not None:
        try:
            seed_value = int(seed)
            if seed_value >= 0:
                return seed_value
        except (TypeError, ValueError):
            pass

    return random.randint(0, 2**31 - 1)


def run_generate(**kwargs):
    """
    Central generation wrapper.

    Current VoxCPM2 returns a waveform directly from model.generate().
    collect_audio() also supports iterator/tensor outputs for compatibility.
    """
    try:
        with torch.inference_mode():
            result = model.generate(**kwargs)
        return collect_audio(result)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise gr.Error(
            "GPU out of memory. Try fewer inference steps, shorter text, "
            "or disable the denoiser in the model settings."
        )
    except Exception as e:
        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}")


# ── Tab 1: Text-to-Speech ─────────────────────────────────────────

def tts_generate(text, cfg, steps, seed, locked):
    if not text or not text.strip():
        raise gr.Error("Please enter some text.")

    used_seed = resolve_seed(seed, locked)

    wav = run_generate(
        text=text.strip(),
        cfg_value=float(cfg),
        inference_timesteps=int(steps),
        seed=used_seed,
    )

    return save_wav(wav), used_seed


# ── Tab 2: Voice Design ──────────────────────────────────────────

def voice_design(description, text, cfg, steps, seed, locked):
    if not text or not text.strip():
        raise gr.Error("Please enter text content.")

    if not description or not description.strip():
        raise gr.Error("Please enter a voice description.")

    combined = f"({description.strip()}){text.strip()}"
    used_seed = resolve_seed(seed, locked)

    wav = run_generate(
        text=combined,
        cfg_value=float(cfg),
        inference_timesteps=int(steps),
        seed=used_seed,
    )

    return save_wav(wav), used_seed


# ── Tab 3: Voice Cloning ─────────────────────────────────────────

def voice_clone(text, ref_audio, style, cfg, steps, seed, locked):
    if not text or not text.strip():
        raise gr.Error("Please enter text.")

    if ref_audio is None:
        raise gr.Error("Please upload reference audio.")

    if style and style.strip():
        text = f"({style.strip()}){text.strip()}"
    else:
        text = text.strip()

    used_seed = resolve_seed(seed, locked)

    wav = run_generate(
        text=text,
        reference_wav_path=ref_audio,
        cfg_value=float(cfg),
        inference_timesteps=int(steps),
        seed=used_seed,
    )

    return save_wav(wav), used_seed


# ── Tab 4: Ultimate Cloning ──────────────────────────────────────

def ultimate_clone(text, ref_audio, transcript, cfg, steps, seed, locked):
    if not text or not text.strip():
        raise gr.Error("Please enter text.")

    if ref_audio is None:
        raise gr.Error("Please upload reference audio.")

    if not transcript or not transcript.strip():
        raise gr.Error("Please enter the reference audio transcript.")

    used_seed = resolve_seed(seed, locked)

    wav = run_generate(
        text=text.strip(),
        prompt_wav_path=ref_audio,
        prompt_text=transcript.strip(),
        reference_wav_path=ref_audio,
        cfg_value=float(cfg),
        inference_timesteps=int(steps),
        seed=used_seed,
    )

    return save_wav(wav), used_seed


# ── Seed UI ───────────────────────────────────────────────────────

def seed_row():
    with gr.Row():
        seed = gr.Number(value=-1, label="Seed", precision=0, scale=3)
        locked = gr.Checkbox(value=False, label="🔒 Lock Seed", scale=1)
    return seed, locked


# ── Build UI ──────────────────────────────────────────────────────

with gr.Blocks(theme=gr.themes.Soft(), css=CSS) as demo:
    gr.HTML(BRAND_HTML)

    with gr.Tab("🗣️ Text-to-Speech"):
        with gr.Accordion("📖 Instructions", open=False):
            gr.Markdown(
                "Enter any text and the model will synthesize speech with natural prosody.\n\n"
                "- **CFG Scale**: Higher = more adherence to text, lower = more natural/relaxed\n"
                "- **Inference Steps**: Higher = better quality but slower (try 6-10 for speed)\n"
                "- **Seed**: Shows the seed used. Check **🔒 Lock Seed** to reuse it for reproducible results.\n"
                "- **Supported Languages (30):** Arabic, Burmese, Chinese, Danish, Dutch, English, "
                "Finnish, French, German, Greek, Hebrew, Hindi, Indonesian, Italian, Japanese, Khmer, "
                "Korean, Lao, Malay, Norwegian, Polish, Portuguese, Russian, Spanish, Swahili, Swedish, "
                "Tagalog, Thai, Turkish, Vietnamese\n"
                "- **Chinese Dialects:** 四川话, 粤语, 吴语, 东北话, 河南话, 陕西话, 山东话, 天津话, 闽南话"
            )

        with gr.Row():
            with gr.Column():
                tts_text = gr.Textbox(
                    label="Text",
                    lines=4,
                    placeholder="Enter text in any of the 30 supported languages..."
                )

                with gr.Row():
                    tts_cfg = gr.Slider(
                        0.5, 5.0, value=2.8, step=0.1, label="CFG Scale"
                    )
                    tts_steps = gr.Slider(
                        5, 30, value=15, step=1, label="Inference Steps"
                    )

                tts_seed, tts_locked = seed_row()
                tts_btn = gr.Button(
                    "🗣️ Generate Speech", variant="primary", size="lg"
                )

            with gr.Column():
                tts_out = gr.Audio(label="Output", type="filepath")

        tts_btn.click(
            tts_generate,
            [tts_text, tts_cfg, tts_steps, tts_seed, tts_locked],
            [tts_out, tts_seed],
        )


    with gr.Tab("🎨 Voice Design"):
        with gr.Accordion("📖 Instructions", open=False):
            gr.Markdown(
                "Create a brand-new voice from a natural-language description — no reference audio needed!\n\n"
                "**How it works:** Describe the voice you want (gender, age, tone, emotion, pace, accent) "
                "and the model will create it.\n\n"
                "**Examples:**\n"
                "- `A young woman, gentle and sweet voice`\n"
                "- `An elderly British man, deep and authoritative`\n"
                "- `A cheerful child, energetic and playful`\n"
                "- `A calm female narrator with a warm tone`\n\n"
                "💡 *Results may vary between runs — try 1-3 times to get the desired voice.*\n"
                "🔒 *Lock the seed to reproduce a voice you liked.*"
            )

        with gr.Row():
            with gr.Column():
                vd_desc = gr.Textbox(
                    label="Voice Description",
                    lines=2,
                    placeholder="A young woman, gentle and sweet voice"
                )
                vd_text = gr.Textbox(
                    label="Text Content",
                    lines=3,
                    placeholder="Hello, welcome to VoxCPM2!"
                )

                with gr.Row():
                    vd_cfg = gr.Slider(
                        0.5, 5.0, value=2.8, step=0.1, label="CFG Scale"
                    )
                    vd_steps = gr.Slider(
                        5, 30, value=15, step=1, label="Inference Steps"
                    )

                vd_seed, vd_locked = seed_row()
                vd_btn = gr.Button(
                    "🎨 Design Voice", variant="primary", size="lg"
                )

            with gr.Column():
                vd_out = gr.Audio(label="Output", type="filepath")

        vd_btn.click(
            voice_design,
            [vd_desc, vd_text, vd_cfg, vd_steps, vd_seed, vd_locked],
            [vd_out, vd_seed],
        )


    with gr.Tab("🎛️ Voice Cloning"):
        with gr.Accordion("📖 Instructions", open=False):
            gr.Markdown(
                "Upload a short reference audio clip to clone the voice.\n\n"
                "- The model clones **timbre, accent, and speaking style** from the reference\n"
                "- Optionally add a **style description** to control emotion/pace while preserving timbre\n"
                "- Reference audio should be **clear and 5-30 seconds** for best results\n"
                "- The model accepts **16kHz input** and outputs **48kHz audio**\n\n"
                "**Style control examples:**\n"
                "- `slightly faster, cheerful tone`\n"
                "- `slow and dramatic`\n"
                "- `whispering, intimate`"
            )

        with gr.Row():
            with gr.Column():
                vc_ref = gr.Audio(
                    label="Reference Audio",
                    type="filepath"
                )
                vc_text = gr.Textbox(
                    label="Text to Synthesize",
                    lines=3,
                    placeholder="This is a cloned voice generated by VoxCPM2."
                )
                vc_style = gr.Textbox(
                    label="Style Description (optional)",
                    lines=1,
                    placeholder="slightly faster, cheerful tone"
                )

                with gr.Row():
                    vc_cfg = gr.Slider(
                        0.5, 5.0, value=2.8, step=0.1, label="CFG Scale"
                    )
                    vc_steps = gr.Slider(
                        5, 30, value=15, step=1, label="Inference Steps"
                    )

                vc_seed, vc_locked = seed_row()
                vc_btn = gr.Button(
                    "🎛️ Clone Voice", variant="primary", size="lg"
                )

            with gr.Column():
                vc_out = gr.Audio(label="Output", type="filepath")

        vc_btn.click(
            voice_clone,
            [vc_text, vc_ref, vc_style, vc_cfg, vc_steps, vc_seed, vc_locked],
            [vc_out, vc_seed],
        )


    with gr.Tab("🎙️ Ultimate Cloning"):
        with gr.Accordion("📖 Instructions", open=False):
            gr.Markdown(
                "Provide reference audio **and** its exact transcript for maximum fidelity cloning.\n\n"
                "- This mode reproduces **timbre, rhythm, emotion, and style** from the reference\n"
                "- The model continues seamlessly from the reference audio\n"
                "- **Both** the reference audio and its transcript are required\n"
                "- For best results, use the **same audio file** as both reference and prompt\n\n"
                "💡 *This is the highest quality cloning mode — use it when you need maximum voice similarity.*"
            )

        with gr.Row():
            with gr.Column():
                uc_ref = gr.Audio(
                    label="Reference Audio",
                    type="filepath"
                )
                uc_transcript = gr.Textbox(
                    label="Reference Audio Transcript",
                    lines=2,
                    placeholder="The exact transcript of the reference audio."
                )
                uc_text = gr.Textbox(
                    label="Text to Synthesize",
                    lines=3,
                    placeholder="This is an ultimate cloning demonstration."
                )

                with gr.Row():
                    uc_cfg = gr.Slider(
                        0.5, 5.0, value=2.8, step=0.1, label="CFG Scale"
                    )
                    uc_steps = gr.Slider(
                        5, 30, value=15, step=1, label="Inference Steps"
                    )

                uc_seed, uc_locked = seed_row()
                uc_btn = gr.Button(
                    "🎙️ Ultimate Clone", variant="primary", size="lg"
                )

            with gr.Column():
                uc_out = gr.Audio(label="Output", type="filepath")

        uc_btn.click(
            ultimate_clone,
            [uc_text, uc_ref, uc_transcript, uc_cfg, uc_steps, uc_seed, uc_locked],
            [uc_out, uc_seed],
        )


# Launch with share=True for a public Colab link
demo.queue().launch(share=True, debug=True)


---

<div align="center">

  <a href="https://www.youtube.com/@TheBlackBoxSecurity?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/TheBlackBoxSecurity">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://TheBlackBoxSecurity.com">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>The BlackBox Security</strong> · TheBlackBoxSecurity.com · © All rights reserved
</p>

---